In [ ]:
import sys
import os
import random
import torch

repo_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
sys.path.insert(0, repo_root)

from vae_generator import VAEGenerator

SEQ_LENGTH = 10
INPUT_DIM = SEQ_LENGTH * 20  # 120
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")

# --- Generate temporary peptide data ---
random.seed(42)
peptides = list(set(
    ''.join(random.choices(AMINO_ACIDS, k=SEQ_LENGTH))
    for _ in range(500)
))
print(f"Generated {len(peptides)} unique temporary peptides")
print(f"Example sequences: {peptides[:5]}")

# --- One-hot encode ---
aa_index = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

def encode(sequences, seq_length):
    tensor = torch.zeros(len(sequences), seq_length * 20)
    for i, seq in enumerate(sequences):
        for j, aa in enumerate(seq):
            tensor[i, j * 20 + aa_index[aa]] = 1.0
    return tensor

one_hot = encode(peptides, SEQ_LENGTH)
print(f"Tensor shape: {one_hot.shape}")  # Expected: [n, 120]

# --- Initialize model ---
vae = VAEGenerator(input_dim=INPUT_DIM, latent_dim=32, hidden_dim=128)
print(f"Device:     {vae.device}")
print(f"Input dim:  {vae.input_dim}")
print(f"Latent dim: {vae.latent_dim}")
print(f"Hidden dim: {vae.hidden_dim}")

# --- Train ---
one_hot = one_hot.to(vae.device)
vae.train_model(one_hot, epochs=300, batch_size=64, lr=1e-3)
print("Training complete!")

# # --- Generate ---
# generated = vae.generate_peptides(count=10)
amino_acid_set = set(AMINO_ACIDS)
unique_generated = set()
# for i, pep in enumerate(generated):
#     valid = all(c in amino_acid_set for c in pep) and len(pep) == SEQ_LENGTH
#     status = "✓" if valid else "✗ INVALID"
#     unique_generated.add(pep)
#     print(f"Peptide {i+1:02d}: {pep}  {status}")

# Generate from seed
seed_peptide = peptides[0]
generated_seed = vae.generate_from_seed(seed_peptide, count=50, exploration=0.4,temperature=0.8)
print(f"\nGenerated from seed '{seed_peptide}':")
for i, pep in enumerate(generated_seed):
    valid = all(c in amino_acid_set for c in pep) and len(pep) == SEQ_LENGTH
    status = "✓" if valid else "✗ INVALID"
    print(f"Peptide {i+1:02d}: {pep}  {status}")
# # --- Novelty + diversity check ---
# training_set = set(peptides)
# novel = [p for p in generated if p not in training_set]
# print(f"\nNovel peptides  (not in training data): {len(novel)}/{len(generated)}")
# print(f"Unique peptides (diversity check):      {len(unique_generated)}/{len(generated)}")


Generated 500 unique temporary peptides
Example sequences: ['GGATWQEKHN', 'MKDIEPQPYN', 'AICNHIGICW', 'AFMFMGLRDQ', 'PGKVINYKEA']
Tensor shape: torch.Size([500, 200])
Device:     cuda
Input dim:  200
Latent dim: 32
Hidden dim: 128


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


  Epoch 050/300 | Recon: 1.2078 | KL: 0.8262 | KL weight: 0.49
  Epoch 100/300 | Recon: 0.7633 | KL: 0.7044 | KL weight: 0.99
  Epoch 150/300 | Recon: 0.5271 | KL: 0.7094 | KL weight: 1.00
  Epoch 200/300 | Recon: 0.4137 | KL: 0.7087 | KL weight: 1.00
  Epoch 250/300 | Recon: 0.3669 | KL: 0.6995 | KL weight: 1.00
  Epoch 300/300 | Recon: 0.3716 | KL: 0.6986 | KL weight: 1.00
Training complete!
Peptide 01: VSSQLECRVM  ✓
Peptide 02: GALYRQIKEG  ✓
Peptide 03: VHMFLDVHSL  ✓
Peptide 04: WQGHAHKFRY  ✓
Peptide 05: ACALCLWTHN  ✓
Peptide 06: TPTHEGWDNR  ✓
Peptide 07: YFEFNEIYQG  ✓
Peptide 08: FCKAIQEELD  ✓
Peptide 09: PRMCKCLWTY  ✓
Peptide 10: RVGTQHADHE  ✓

Generated from seed 'GGATWQEKHN':
Peptide 01: GGGTWQTKHN  ✓
Peptide 02: CGATRQEKHN  ✓
Peptide 03: GGTTWQEKHN  ✓
Peptide 04: GGTTWQTKHN  ✓
Peptide 05: GGATWQATHN  ✓
Peptide 06: GGATWQWKHN  ✓
Peptide 07: GGATWQNKHN  ✓
Peptide 08: GGATWQTKHN  ✓
Peptide 09: GGATWQAVHN  ✓
Peptide 10: GGATWQEKHN  ✓

Novel peptides  (not in training data): 10/10
U